# 02. Scaffold-tuner v7

**candidate_rank definition (aligned with all other methods)**:
`candidate_rank` is assigned to ALL valid, scaffold-retained, non-duplicate
candidates — BEFORE descriptor delta filtering.  Descriptor success/strict-success
is then evaluated by `make_candidate_record()`.

```
Scaffold-tuner generation (Tier 1 + Tier 2)
        ↓
RDKit build validation
        ↓
scaffold retention check
        ↓
duplicate removal
        ↓
rank by collateral_change, then pool order
        ↓  candidate_rank = 1, 2, 3, ...  ← assigned here
whole-molecule descriptor delta
        ↓
success / strict_success (evaluated in make_candidate_record)
```

This is the same pipeline as Random, CReM, and LibInvent.
The public `propose_structures()` API is NOT used here because it filters
to success-only candidates before ranking.


In [ ]:
import os, sys
import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")

sys.path.append("..")

from configs.benchmark_config import (
    ACTIVE_PARENT_DATASET, RESULTS_DIR,
    N_CANDIDATES_PER_PARENT, RANDOM_SEED, TASKS, TASK_NAMES,
)
from utils.parents import prepare_parent
from utils.records import make_candidate_record

# Benchmark-specific wrapper: returns ALL scaffold-retained candidates
# before descriptor delta filtering
from utils.scaffold_tuner_benchmark import generate_all_candidates

os.makedirs(RESULTS_DIR, exist_ok=True)
METHOD = "scaffold_tuner"


In [ ]:
df = pd.read_csv(ACTIVE_PARENT_DATASET)
if "canonical_smiles" not in df.columns and "smiles" in df.columns:
    df = df.rename(columns={"smiles": "canonical_smiles"})
if "parent_id" not in df.columns and "chembl_id" in df.columns:
    df = df.rename(columns={"chembl_id": "parent_id"})
print(f"Parents: {len(df)}")
df.head(2)


## Generate candidates — all 7 tasks

In [ ]:
records, gen_stats = [], []

for task_name, task_cfg in tqdm(TASKS.items(), desc="tasks"):
    target = task_cfg["descriptor"].upper()
    delta  = task_cfg["delta"]

    for _, row in df.iterrows():
        prepared = prepare_parent(row)
        if prepared is None:
            continue
        parent_id, parent_smiles, parent_desc, _ = prepared

        try:
            candidates, stats = generate_all_candidates(
                parent_smiles,
                target=target,
                delta=delta,
                max_candidates=N_CANDIDATES_PER_PARENT,
                random_seed=RANDOM_SEED,
            )
        except Exception as e:
            candidates, stats = [], {
                "n_examined": 0, "n_invalid": 0,
                "n_scaffold_failed": 0, "n_duplicate": 0, "n_valid_unique": 0,
            }

        gen_stats.append({
            "method": METHOD, "task": task_name, "parent_id": parent_id,
            **stats,
        })

        for cand in candidates:
            rec = make_candidate_record(
                method=METHOD,
                parent_id=parent_id,
                parent_smiles=parent_smiles,
                task_name=task_name,
                candidate_rank=cand["candidate_rank"],
                candidate_smiles=cand["generated_smiles"],
                parent_desc=parent_desc,
                generation_strategy=cand.get("generation_strategy"),
            )
            if rec is not None:
                records.append(rec)

print(f"Total records: {len(records)}")


In [ ]:
OUTPUT_FILE = f"{RESULTS_DIR}/scaffold_tuner.csv"
STATS_FILE  = f"{RESULTS_DIR}/scaffold_tuner_generation_stats.csv"

pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
pd.DataFrame(gen_stats).to_csv(STATS_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
print(f"Saved → {STATS_FILE}")
